# Pneumonia Detection using VGG16 & PyTorch

This notebook covers the complete end-to-end pipeline for pneumonia classification using chest X-ray images:
1. System & GPU Environment Verification
2. Dataset Loading & Preprocessing
3. Transfer Learning Model Setup (VGG16)
4. Model Training & Validation Loop
5. Test Evaluation & Performance Metrics

In [1]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: NVIDIA GeForce RTX 2050


## 1. Dataset Configuration & Transforms

We define relative paths to step out of the `notebooks/` directory and access `full_data/`. Standard ImageNet normalization parameters are applied to match VGG16 pre-training expectations. The dataset is split into 80% training, 10% validation, and 10% test sets.

In [3]:
FULL_DATA_DIR = '../full_data'

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

full_dataset_train = datasets.ImageFolder(root=FULL_DATA_DIR, transform=train_transforms)
full_dataset_eval  = datasets.ImageFolder(root=FULL_DATA_DIR, transform=val_test_transforms)

total_size = len(full_dataset_eval)
train_size = int(0.8 * total_size)
val_size   = int(0.1 * total_size)
test_size  = total_size - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_subset, val_subset, test_subset = random_split(
    range(total_size),
    [train_size, val_size, test_size],
    generator=generator
)

train_dataset = torch.utils.data.Subset(full_dataset_train, train_subset.indices)
val_dataset   = torch.utils.data.Subset(full_dataset_eval, val_subset.indices)
test_dataset  = torch.utils.data.Subset(full_dataset_eval, test_subset.indices)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

class_names = full_dataset_train.classes
print(f"Classes found: {class_names}")
print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)} | Test samples: {len(test_dataset)}")

Classes found: ['NORMAL', 'PNEUMONIA']
Train samples: 4684 | Val samples: 585 | Test samples: 587


## 2. Model Initialization (VGG16 Transfer Learning)

We freeze feature extraction convolutional layers and update the final linear classifier to perform 2-class binary classification (`NORMAL` vs `PNEUMONIA`).

In [4]:
# Load Pre-trained VGG16
weights = models.VGG16_Weights.DEFAULT
model = models.vgg16(weights=weights)

# Freeze convolutional base
for param in model.features.parameters():
    param.requires_grad = False

# Replace final output layer
model.classifier[6] = nn.Linear(model.classifier[6].in_features, len(class_names))

# Send model to GPU
model = model.to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.0001)

## 3. Training & Validation Loop

Train the classifier while preserving and saving the best checkpoint based on validation set accuracy.

In [5]:
EPOCHS = 10
best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

# Main Epochs Progress Bar
epoch_pbar = tqdm(range(EPOCHS), desc="Total Progress", unit="epoch")

for epoch in epoch_pbar:
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print("-" * 25)

    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    running_corrects = 0

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in train_pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)

        loss.backward()
        optimizer.step()

        batch_loss = loss.item() * inputs.size(0)
        batch_corrects = torch.sum(preds == labels.data)

        running_loss += batch_loss
        running_corrects += batch_corrects

        # Live batch update in progress bar
        train_pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "batch_acc": f"{(batch_corrects.double() / inputs.size(0)) * 100:.1f}%"
        })

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)
    print(f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc * 100:.2f}%")

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    val_corrects = 0

    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch + 1}/{EPOCHS} [Val]", leave=False)
    with torch.no_grad():
        for inputs, labels in val_pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

            val_pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    val_epoch_loss = val_loss / len(val_dataset)
    val_epoch_acc = val_corrects.double() / len(val_dataset)
    print(f"Val Loss:   {val_epoch_loss:.4f} | Val Acc:   {val_epoch_acc * 100:.2f}%")

    # --- Save Best Weights ---
    if val_epoch_acc > best_acc:
        best_acc = val_epoch_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    # Update total progress bar description with best val accuracy so far
    epoch_pbar.set_postfix({"Best Val Acc": f"{best_acc * 100:.2f}%"})

# Load best model weights and save
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), 'vgg16_chest_xray.pth')
print("\nTraining completed! Saved best model weights to 'vgg16_chest_xray.pth'.")

Total Progress:   0%|          | 0/10 [00:00<?, ?epoch/s]


Epoch 1/10
-------------------------


Epoch 1/10 [Train]:   0%|          | 0/147 [00:00<?, ?it/s]

Train Loss: 0.1970 | Train Acc: 92.06%


Epoch 1/10 [Val]:   0%|          | 0/19 [00:00<?, ?it/s]

...

Val Loss:   0.0851 | Val Acc:   96.92%

Epoch 10/10
-------------------------


Epoch 10/10 [Train]:   0%|          | 0/147 [00:00<?, ?it/s]

Train Loss: 0.0777 | Train Acc: 97.18%


Epoch 10/10 [Val]:   0%|          | 0/19 [00:00<?, ?it/s]

Val Loss:   0.1444 | Val Acc:   94.36%

Training completed! Saved best model weights to 'vgg16_chest_xray.pth'.


## 4. Test Dataset Evaluation

Evaluating model accuracy on unseen test data and generating full classification metrics (Precision, Recall, F1-Score, and Confusion Matrix).

In [6]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.92      0.93      0.92       162
   PNEUMONIA       0.97      0.97      0.97       425

    accuracy                           0.96       587
   macro avg       0.94      0.95      0.95       587
weighted avg       0.96      0.96      0.96       587

Confusion Matrix:
[[151  11]
 [ 14 411]]
